In [ ]:
# =========================================================
# SIMPLE NL2SQL PROJECT
# SQLITE + GEMINI API (google-genai)
# =========================================================

import os
import re
import sqlite3
from google import genai


# =========================================================
# 1. CREATE GEMINI CLIENT
# =========================================================

api_key = os.environ.get("GEMINI_API_KEY", "").strip()
if not api_key:
    api_key = input("Enter your GEMINI_API_KEY: ").strip()

if not api_key:
    raise ValueError("GEMINI_API_KEY is required to run this script.")

client = genai.Client(api_key=api_key)
MODEL_ID = "gemini-3.8-flash"


# =========================================================
# 2. CREATE SQLITE DATABASE
# =========================================================

conn = sqlite3.connect("sales_nl2sql.db")
cursor = conn.cursor()

print("=" * 60)
print("NL2SQL WITH SQLITE & GEMINI")
print("=" * 60)


# =========================================================
# 3. CREATE SALES TABLE
# =========================================================

cursor.execute("""
CREATE TABLE IF NOT EXISTS sales (
    SaleID INTEGER PRIMARY KEY,
    ProductName TEXT,
    Category TEXT,
    City TEXT,
    Quantity INTEGER,
    Amount REAL,
    SaleDate TEXT,
    Status TEXT
)
""")


# =========================================================
# 4. INSERT 10 SALES RECORDS
# =========================================================

sales_data = [
    (1, "Laptop", "Electronics", "Chennai", 2, 120000, "2026-01-10", "Completed"),
    (2, "Mobile", "Electronics", "Coimbatore", 3, 75000, "2026-01-12", "Completed"),
    (3, "Headphones", "Accessories", "Bengaluru", 5, 25000, "2026-01-15", "Completed"),
    (4, "Tablet", "Electronics", "Chennai", 2, 60000, "2026-01-20", "Cancelled"),
    (5, "Keyboard", "Accessories", "Madurai", 4, 12000, "2026-01-22", "Completed"),
    (6, "Monitor", "Electronics", "Coimbatore", 3, 45000, "2026-02-02", "Completed"),
    (7, "Mouse", "Accessories", "Chennai", 10, 10000, "2026-02-05", "Completed"),
    (8, "Printer", "Electronics", "Bengaluru", 2, 30000, "2026-02-10", "Completed"),
    (9, "Laptop", "Electronics", "Madurai", 1, 65000, "2026-02-15", "Completed"),
    (10, "Mobile", "Electronics", "Chennai", 4, 100000, "2026-02-20", "Completed"),
]

cursor.executemany("""
INSERT OR IGNORE INTO sales
(SaleID, ProductName, Category, City, Quantity, Amount, SaleDate, Status)
VALUES (?, ?, ?, ?, ?, ?, ?, ?)
""", sales_data)

conn.commit()
print("\nSales table ready (up to 10 sample records).")


# =========================================================
# 5. DISPLAY DATABASE RECORDS
# =========================================================

print("\nDATABASE RECORDS")
print("-" * 60)

cursor.execute("SELECT * FROM sales")
for row in cursor.fetchall():
    print(row)


# =========================================================
# 6. GET DATABASE SCHEMA
# =========================================================

cursor.execute("PRAGMA table_info(sales)")
schema_rows = cursor.fetchall()

schema = "TABLE: sales\n\nCOLUMNS:\n"
for row in schema_rows:
    column_name = row[1]
    data_type = row[2]
    schema += f"- {column_name} ({data_type})\n"

print("\nDATABASE SCHEMA")
print("-" * 60)
print(schema)


# =========================================================
# 7. CLEAN & VALIDATE SQL
# =========================================================

def clean_sql(raw_sql: str) -> str:
    cleaned = (raw_sql or "").strip()
    cleaned = re.sub(r"^```(?:sql)?\s*", "", cleaned, flags=re.IGNORECASE)
    cleaned = re.sub(r"\s*```$", "", cleaned)
    return cleaned.strip()


def validate_sql(sql: str) -> bool:
    sql_clean = sql.strip()
    if not sql_clean:
        return False

    # Allow at most one trailing semicolon; block stacked statements
    without_trailing_semi = sql_clean[:-1].strip() if sql_clean.endswith(";") else sql_clean
    if ";" in without_trailing_semi:
        return False

    sql_upper = without_trailing_semi.upper()
    if not re.match(r"^SELECT\b", sql_upper):
        return False

    dangerous_commands = [
        "INSERT", "UPDATE", "DELETE", "DROP", "ALTER",
        "CREATE", "REPLACE", "ATTACH", "DETACH", "PRAGMA"
    ]
    for command in dangerous_commands:
        if re.search(rf"\b{command}\b", sql_upper):
            return False

    return True


# =========================================================
# 8. GENERATE SQL USING GEMINI
# =========================================================

def generate_sql(question: str) -> str:
    prompt = f"""You are an expert SQLite SQL generator.
Convert the user's natural-language question into a single SQLite SELECT query.

DATABASE SCHEMA:
{schema}

RULES:
1. Generate only a SELECT query.
2. Use only the sales table and columns listed in the schema.
3. Do not generate INSERT, UPDATE, DELETE, DROP, ALTER, CREATE, REPLACE, ATTACH, DETACH, or PRAGMA.
4. Return ONLY raw SQL with no markdown formatting and no explanations.

USER QUESTION:
{question}
"""
    response = client.models.generate_content(
        model=MODEL_ID,
        contents=prompt,
    )
    return clean_sql(response.text)


# =========================================================
# 9. EXECUTE GENERATED SQL
# =========================================================

def execute_sql(sql: str) -> None:
    print("\nGENERATED SQL")
    print("-" * 60)
    print(sql)

    if not validate_sql(sql):
        print("\nUnsafe or invalid SQL rejected.")
        return

    try:
        cursor.execute(sql)
        results = cursor.fetchall()
        columns = [desc[0] for desc in cursor.description] if cursor.description else []

        print("\nQUERY RESULT")
        print("-" * 60)
        if results:
            if columns:
                print(" | ".join(columns))
                print("-" * 60)
            for row in results:
                print(row)
        else:
            print("No records found.")

    except sqlite3.Error as e:
        print("\nSQLite Error:", e)


# =========================================================
# 10. ASK USER QUESTIONS IN A LOOP
# =========================================================

print("\nType 'exit' or 'quit' to stop.")
try:
    while True:
        question = input("\nAsk a question about the sales data: ").strip()
        if not question:
            continue
        if question.lower() in {"exit", "quit"}:
            break

        print("\nYour question:", question)
        try:
            generated_sql = generate_sql(question)
            execute_sql(generated_sql)
        except Exception as e:
            print("\nError generating or running SQL:", e)
finally:
    conn.close()
    print("\nDatabase connection closed.")